# Connecting with ADLS Gen2

In [0]:
%run /Workspace/Repos/azbhaskar18@outlook.com/adfrepo-aug/Notebooks/Connectivitity

### Creating database

In [0]:
%sql
create database if not exists hive_metastore.silver_db

In [0]:
%sql
use silver_db

### Reading data from Bronze to Silver and applying transformation

In [0]:
%sql
SELECT * FROM bronze_db.calendar LIMIT 5

Date
1/1/2015
1/2/2015
1/3/2015
1/4/2015
1/5/2015


In [0]:
dbutils.widgets.text("stoarge_account_name", "bhaskaradlsaug")
stoarge_account_name=dbutils.widgets.get("stoarge_account_name")
print(stoarge_account_name)

bhaskaradlsaug


In [0]:
#
from pyspark.sql.functions import *
df1=spark.sql("""
            SELECT * FROM bronze_db.calendar
          """)
df2=df1.withColumn("Date", to_date("Date", "m/d/yyyy"))
finalres=df2.withColumn("MonthNbr", month(col("Date"))) \
            .withColumn("Month", date_format(col("Date"), "MMM")) \
            .withColumn("Year", year(col("Date"))) 
finalres.write.format("delta").mode("append").save(f"abfss://silver@{stoarge_account_name}.dfs.core.windows.net/calendar/")

Date,MonthNbr,Month,Year
2015-01-01,1,Jan,2015
2015-01-02,1,Jan,2015
2015-01-03,1,Jan,2015
2015-01-04,1,Jan,2015
2015-01-05,1,Jan,2015
2015-01-06,1,Jan,2015
2015-01-07,1,Jan,2015
2015-01-08,1,Jan,2015
2015-01-09,1,Jan,2015
2015-01-10,1,Jan,2015


In [0]:
spark.sql(f"""
create table if not exists silver_db.calendar (
    Date date,
    MonthNbr int,
    Month string,
    Year int
)
location 'abfss://silver@{stoarge_account_name}.dfs.core.windows.net/calendar'
""")

#### Category table

In [0]:
spark.sql(f"""
create table if not exists silver_db.category (
    CategoryKey int,
    Category string
)
location 'abfss://silver@{stoarge_account_name}.dfs.core.windows.net/category'
""")

In [0]:
df1=spark.sql("""
            SELECT * FROM bronze_db.category
          """)
df2=df1.withColumnRenamed("ProductCategoryKey", "CategoryKey") \
  .withColumnRenamed("CategoryName", "Category").distinct() \
  .withColumn("CategoryKey", col("CategoryKey").cast("int"))
df2.write.format("delta").mode("overwrite").save(f"abfss://silver@{stoarge_account_name}.dfs.core.windows.net/category")

#### Creting Subcategory table

In [0]:
spark.sql(f"""
create table if not exists silver_db.subcategory (
    SubcategoryKey int,
    Subcategory string,
    CategoryKey int
)
location 'abfss://silver@{stoarge_account_name}.dfs.core.windows.net/subcategory'
""")

In [0]:
df1=spark.sql("""
            SELECT * FROM bronze_db.subcategory
          """)
df2=df1.withColumnRenamed("ProductSubcategoryKey", "SubcategoryKey") \
        .withColumnRenamed("SubcategoryName", "Subcategory") \
        .withColumnRenamed("ProductCategoryKey", "CategoryKey")

finalres=df2.withColumn("CategoryKey", col("CategoryKey").cast("int")) \
        .withColumn("SubcategoryKey", col("SubcategoryKey").cast("int"))

finalres.write.format("delta").mode("append").save(f"abfss://silver@{stoarge_account_name}.dfs.core.windows.net/subcategory")

#### Product table

In [0]:
spark.sql(f"""
create table if not exists silver_db.product (
    ProductKey int,
    Product string,
    SubcategoryKey int
)
location 'abfss://silver@{stoarge_account_name}.dfs.core.windows.net/product'
""")

In [0]:
df1=spark.sql("""
            SELECT * FROM bronze_db.product
          """)
df2=df1.select("ProductKey", "ProductName", "ProductSubcategoryKey")
df3=df2.withColumnRenamed("ProductName", "Product").withColumnRenamed("ProductSubcategoryKey", "SubcategoryKey")
finalres=df3.withColumn("ProductKey", col("ProductKey").cast("int")).withColumn("SubcategoryKey", col("SubcategoryKey").cast("int"))
finalres.write.format("delta").mode("append").save(f"abfss://silver@{stoarge_account_name}.dfs.core.windows.net/product")

#### Creating Sales

In [0]:
spark.sql(f"""
create table if not exists silver_db.sales (
    OrderDate date,
    OrderNumber string,
    ProductKey int,
    OrderLineItem int,
    OrderQuantity int
)
location 'abfss://silver@{stoarge_account_name}.dfs.core.windows.net/sales'
""")

In [0]:
df1=spark.sql("""
            SELECT * FROM bronze_db.sales
          """)
df2=df1.withColumn("OrderDate", to_date("OrderDate", "m/d/yyyy")) \
    .withColumn("OrderQuantity", col("OrderQuantity").cast("int")) \
    .withColumn("OrderLineItem", col("OrderLineItem").cast("int")) \
    .withColumn("ProductKey", col("ProductKey").cast("int")) \
    .select("OrderDate", "OrderNumber", "ProductKey",   "OrderLineItem", "OrderQuantity")
df2.write.format("delta").mode("append").save(f"abfss://silver@{stoarge_account_name}.dfs.core.windows.net/sales")